In [ ]:

"""
SHARED SETUP - Run ONCE at the beginning
"""

print("\n" + "="*80)
print("SETUP: DEPENDENCIES & DATA")
print("="*80)

# Install dependencies
!pip install -q transformers peft timm accelerate torch torchvision segmentation-models-pytorch scikit-learn pandas matplotlib seaborn

# Mount drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/harshini2806/lora_paper.git
%cd lora_paper

import os
os.makedirs('/content/drive/MyDrive/lora_experiments', exist_ok=True)

# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
import numpy as np
import time
from collections import Counter
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'



# Load data
from dataloader import GenDEBRIS

train_ds = GenDEBRIS(mode='train', path='/content/lora_paper/data')
val_ds = GenDEBRIS(mode='val', path='/content/lora_paper/data')

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# Compute class weights
label_counts = Counter()
for _, mask in train_ds:
    valid = mask[mask != -1].flatten()
    label_counts.update(valid.tolist())

total = sum(label_counts.values())
freqs = np.array([label_counts.get(i, 0)/total for i in range(11)])
weights = 1.0 / (freqs + 1e-6)
weights = weights / weights.sum()
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# Metrics function
def compute_metrics(preds, labels, num_classes=11):
    if isinstance(preds, torch.Tensor):
        preds = torch.argmax(preds, dim=1).cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    valid = labels != -1
    preds_v = preds[valid]
    labels_v = labels[valid]

    accuracy = (preds_v == labels_v).sum() / labels_v.size
    f1 = f1_score(labels_v, preds_v, average='weighted', zero_division=0)

    present_classes = np.unique(labels_v)
    ious = []
    for cls in present_classes:
        pred_m = (preds_v == cls)
        true_m = (labels_v == cls)
        inter = (pred_m & true_m).sum()
        union = (pred_m | true_m).sum()
        if union > 0:
            ious.append(inter / union)
    mean_iou = np.mean(ious) if len(ious) > 0 else 0.0

    return {'Accuracy': accuracy, 'F1': f1, 'mIoU': mean_iou}



# Training function
def train(model, name, epochs=70, lr=None):
    print(f"\n{'#'*70}\n# {name}\n{'#'*70}\n")

    if lr is None:
        lr = 2e-4 if 'lora' in name.lower() else 5e-5

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
    scaler = GradScaler()

    train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)

    best_metrics = {'mIoU':0, 'F1':0, 'Accuracy':0}
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        total_loss = 0

        for imgs, masks in train_dl:
            if imgs.dim()==4 and imgs.shape[-1]==11:
                imgs = imgs.permute(0,3,1,2)
            imgs = F.interpolate(imgs, size=(224,224), mode='bilinear', align_corners=False)
            imgs, masks = imgs.float().to(device), masks.long().to(device)

            opt.zero_grad()
            with autocast("cuda"):
                out = model(imgs)
                if out.shape[2:] != masks.shape[1:]:
                    out = F.interpolate(out, masks.shape[1:], mode='bilinear')
                loss = loss_fn(out, masks)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item()

        scheduler.step()

        # Validation
        model.eval()
        val_metrics = {'mIoU':0, 'F1':0, 'Accuracy':0}
        with torch.no_grad():
            for imgs, masks in val_dl:
                if imgs.dim()==4 and imgs.shape[-1]==11:
                    imgs = imgs.permute(0,3,1,2)
                imgs = F.interpolate(imgs, size=(224,224), mode='bilinear', align_corners=False)
                imgs, masks = imgs.float().to(device), masks.long().to(device)

                with autocast("cuda"):
                    out = model(imgs)
                    if out.shape[2:] != masks.shape[1:]:
                        out = F.interpolate(out, masks.shape[1:], mode='bilinear')

                met = compute_metrics(out, masks)
                for k in val_metrics:
                    val_metrics[k] += met[k]

        for k in val_metrics:
            val_metrics[k] /= len(val_dl)

        print(f"Ep {ep+1:02d} | Loss={total_loss/len(train_dl):.4f} | mIoU={val_metrics['mIoU']:.4f} | "
              f"F1={val_metrics['F1']:.4f} | Acc={val_metrics['Accuracy']:.4f}")

        if val_metrics['mIoU'] > best_metrics['mIoU']:
            best_metrics = val_metrics.copy()
            torch.save(model.state_dict(), f"/content/drive/MyDrive/lora_experiments/{name}_best.pth")

    best_metrics['name'] = name
    best_metrics['time_min'] = (time.time()-t0)/60
    for k in ['mIoU', 'F1', 'Accuracy']:
      best_metrics[k] = float(best_metrics[k])

    return best_metrics

print("✅ SETUP COMPLETE - Ready to train models!")



SETUP: DEPENDENCIES & DATA
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.0 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into 'lora_paper'...
remote: Enumerating objects: 4432, done.
remote: Total 4432 (delta 0), reused 0 (delta 0), pack-reused 4432 (from 1)
Receiving objects: 100% (4432/4432), 1.08 GiB | 19.00 MiB/s, done.
Resolving deltas: 100% (2817/2817), done.
Updating files: 100% (4478/4478), done.
/content/lora_paper


Load train set to memory:   0%|          | 0/694 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
Load val set to memory: 100%|██████████| 328/328 [00:06<00:00, 49.86it/s]


Train: 694, Val: 328
✅ SETUP COMPLETE - Ready to train models!


In [ ]:
from segmentation_models_pytorch import Unet
import torch.nn as nn

def create_unet_baseline():
    print("🟡 Creating UNet Baseline (ResNet50 encoder)")

    model = Unet(
        encoder_name="resnet50",
        encoder_weights="imagenet",
        in_channels=11,           # dataset input channels
        classes=11,               # number of segmentation classes
    )
    return model


In [ ]:
from peft import LoraConfig, get_peft_model

def create_unet_lora(lora_r=16):
    model = create_unet_baseline()
    print(f"🔷 Adding LoRA adapters with r={lora_r}")

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_r*2,
        target_modules=[
            "conv1", "conv2", "conv3", "conv4", "downsample.0"
        ],
        lora_dropout=0.1,
        bias="none",
        modules_to_save=["decoder", "segmentation_head"]  # keeping decoder trainable
    )

    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


In [ ]:
baseline_unet = create_unet_baseline().to(device)
result_baseline = train(baseline_unet, "unet_baseline_70ep", epochs=70, lr=5e-5)
print("✅ BASELINE COMPLETE:", result_baseline)


🟡 Creating UNet Baseline (ResNet50 encoder)

######################################################################
# unet_baseline_70ep
######################################################################

Ep 01 | Loss=2.4265 | mIoU=0.0880 | F1=0.1956 | Acc=0.1477
Ep 02 | Loss=2.0362 | mIoU=0.1992 | F1=0.3976 | Acc=0.3309
Ep 03 | Loss=1.8432 | mIoU=0.2647 | F1=0.4911 | Acc=0.4347
Ep 04 | Loss=1.7152 | mIoU=0.2893 | F1=0.5523 | Acc=0.5110
Ep 05 | Loss=1.6567 | mIoU=0.2682 | F1=0.5169 | Acc=0.4603
Ep 06 | Loss=1.5920 | mIoU=0.3181 | F1=0.5709 | Acc=0.5359
Ep 07 | Loss=1.3852 | mIoU=0.3222 | F1=0.5735 | Acc=0.5338
Ep 08 | Loss=1.3303 | mIoU=0.3279 | F1=0.5739 | Acc=0.5317
Ep 09 | Loss=1.1767 | mIoU=0.3772 | F1=0.6427 | Acc=0.6052
Ep 10 | Loss=1.1742 | mIoU=0.4260 | F1=0.6717 | Acc=0.6405
Ep 11 | Loss=1.0012 | mIoU=0.3700 | F1=0.6303 | Acc=0.5968
Ep 12 | Loss=1.0591 | mIoU=0.4135 | F1=0.6410 | Acc=0.6082
Ep 13 | Loss=0.9186 | mIoU=0.4263 | F1=0.6719 | Acc=0.6351
Ep 14 | Loss=0.8385 | mI

In [ ]:
lora_unet_r4 = create_unet_lora(lora_r=4).to(device)
result_r4 = train(lora_unet_r4, "unet_lora_r4_70ep", epochs=70, lr=5e-5)
print("✅ r=4 COMPLETE:", result_r4)


🟡 Creating UNet Baseline (ResNet50 encoder)
🔷 Adding LoRA adapters with r=4
trainable params: 9,333,863 || all params: 41,881,506 || trainable%: 22.2864

######################################################################
# unet_lora_r4_70ep
######################################################################

Ep 01 | Loss=2.5436 | mIoU=0.0876 | F1=0.2000 | Acc=0.1567
Ep 02 | Loss=2.2696 | mIoU=0.1131 | F1=0.2548 | Acc=0.2129
Ep 03 | Loss=2.1632 | mIoU=0.1278 | F1=0.2691 | Acc=0.2295
Ep 04 | Loss=2.0031 | mIoU=0.1534 | F1=0.3074 | Acc=0.2717
Ep 05 | Loss=1.8811 | mIoU=0.1845 | F1=0.3670 | Acc=0.3248
Ep 06 | Loss=1.7038 | mIoU=0.1988 | F1=0.3814 | Acc=0.3370
Ep 07 | Loss=1.6342 | mIoU=0.2129 | F1=0.3913 | Acc=0.3445
Ep 08 | Loss=1.5248 | mIoU=0.2165 | F1=0.4117 | Acc=0.3613
Ep 09 | Loss=1.3409 | mIoU=0.2750 | F1=0.4995 | Acc=0.4457
Ep 10 | Loss=1.3512 | mIoU=0.2467 | F1=0.4767 | Acc=0.4247
Ep 11 | Loss=1.2084 | mIoU=0.2628 | F1=0.4871 | Acc=0.4439
Ep 12 | Loss=1.1019 | mIoU=0.3033 

In [ ]:
lora_unet_r8 = create_unet_lora(lora_r=8).to(device)
result_r8 = train(lora_unet_r8, "unet_lora_r8_70ep", epochs=70, lr=5e-5)
print("✅ r=8 COMPLETE:", result_r8)


🟡 Creating UNet Baseline (ResNet50 encoder)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

🔷 Adding LoRA adapters with r=8
trainable params: 9,653,203 || all params: 42,200,846 || trainable%: 22.8744

######################################################################
# unet_lora_r8_70ep
######################################################################

Ep 01 | Loss=2.5725 | mIoU=0.0727 | F1=0.1636 | Acc=0.1194
Ep 02 | Loss=2.2137 | mIoU=0.1204 | F1=0.2377 | Acc=0.2111
Ep 03 | Loss=2.0046 | mIoU=0.1321 | F1=0.2514 | Acc=0.2245
Ep 04 | Loss=1.8039 | mIoU=0.1616 | F1=0.3126 | Acc=0.2721
Ep 05 | Loss=1.7334 | mIoU=0.1887 | F1=0.3704 | Acc=0.3160
Ep 06 | Loss=1.5758 | mIoU=0.1891 | F1=0.3510 | Acc=0.3001
Ep 07 | Loss=1.5640 | mIoU=0.1985 | F1=0.3504 | Acc=0.3145
Ep 08 | Loss=1.4758 | mIoU=0.2398 | F1=0.4452 | Acc=0.4187
Ep 09 | Loss=1.3810 | mIoU=0.2625 | F1=0.4531 | Acc=0.4191
Ep 10 | Loss=1.3076 | mIoU=0.2517 | F1=0.4577 | Acc=0.4100
Ep 11 | Loss=1.2443 | mIoU=0.2609 | F1=0.4805 | Acc=0.4421
Ep 12 | Loss=1.1268 | mIoU=0.2578 | F1=0.4819 | Acc=0.4378
Ep 13 | Loss=1.1484

In [ ]:
lora_unet_r16 = create_unet_lora(lora_r=16).to(device)
result_r16 = train(lora_unet_r16, "unet_lora_r16_70ep", epochs=70, lr=5e-5)
print("✅ r=16 COMPLETE:", result_r16)


🟡 Creating UNet Baseline (ResNet50 encoder)
🔷 Adding LoRA adapters with r=16
trainable params: 10,291,883 || all params: 42,839,526 || trainable%: 24.0243

######################################################################
# unet_lora_r16_70ep
######################################################################

Ep 01 | Loss=2.7755 | mIoU=0.1271 | F1=0.2793 | Acc=0.2177
Ep 02 | Loss=2.1886 | mIoU=0.1607 | F1=0.3302 | Acc=0.2860
Ep 03 | Loss=2.0272 | mIoU=0.1939 | F1=0.3654 | Acc=0.3167
Ep 04 | Loss=1.8525 | mIoU=0.2189 | F1=0.3846 | Acc=0.3523
Ep 05 | Loss=1.7310 | mIoU=0.2410 | F1=0.4452 | Acc=0.3922
Ep 06 | Loss=1.5884 | mIoU=0.2730 | F1=0.4696 | Acc=0.4278
Ep 07 | Loss=1.5051 | mIoU=0.2835 | F1=0.5047 | Acc=0.4586
Ep 08 | Loss=1.4040 | mIoU=0.2727 | F1=0.4703 | Acc=0.4223
Ep 09 | Loss=1.2716 | mIoU=0.2733 | F1=0.4625 | Acc=0.4157
Ep 10 | Loss=1.1974 | mIoU=0.2901 | F1=0.4908 | Acc=0.4411
Ep 11 | Loss=1.1432 | mIoU=0.2732 | F1=0.5054 | Acc=0.4601
Ep 12 | Loss=1.0508 | mIoU=0.31

In [ ]:
lora_unet_r32 = create_unet_lora(lora_r=32).to(device)
result_r32 = train(lora_unet_r32, "unet_lora_r32_70ep", epochs=70, lr=5e-5)
print("✅ r=32 COMPLETE:", result_r32)


🟡 Creating UNet Baseline (ResNet50 encoder)
🔷 Adding LoRA adapters with r=32
trainable params: 11,569,243 || all params: 44,116,886 || trainable%: 26.2241

######################################################################
# unet_lora_r32_70ep
######################################################################

Ep 01 | Loss=2.6587 | mIoU=0.0456 | F1=0.0835 | Acc=0.0664
Ep 02 | Loss=2.2383 | mIoU=0.0579 | F1=0.1134 | Acc=0.0914
Ep 03 | Loss=2.0699 | mIoU=0.0907 | F1=0.1818 | Acc=0.1468
Ep 04 | Loss=1.8727 | mIoU=0.1320 | F1=0.2532 | Acc=0.2118
Ep 05 | Loss=1.7747 | mIoU=0.1744 | F1=0.3342 | Acc=0.2912
Ep 06 | Loss=1.6535 | mIoU=0.2027 | F1=0.3752 | Acc=0.3288
Ep 07 | Loss=1.5802 | mIoU=0.1974 | F1=0.3652 | Acc=0.3230
Ep 08 | Loss=1.4738 | mIoU=0.2148 | F1=0.3863 | Acc=0.3434
Ep 09 | Loss=1.3650 | mIoU=0.2370 | F1=0.4156 | Acc=0.3689
Ep 10 | Loss=1.2957 | mIoU=0.2668 | F1=0.4416 | Acc=0.4065
Ep 11 | Loss=1.1838 | mIoU=0.2586 | F1=0.4389 | Acc=0.4030
Ep 12 | Loss=1.1413 | mIoU=0.23